In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

load dataset

In [ ]:
file_path = '/content/drive/MyDrive/razorpay hackathon dataset/online_retail_II.xlsx'
df = pd.read_excel(file_path)

display(df.head())

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [ ]:
df.shape

(525461, 8)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


In [ ]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,525461.000000,525461,525461.000000,417534.000000
mean,10.337667,2010-06-28 11:37:36.845017856,4.688834,15360.645478
min,-9600.000000,2009-12-01 07:45:00,-53594.360000,12346.000000
25%,1.000000,2010-03-21 12:20:00,1.250000,13983.000000
50%,3.000000,2010-07-06 09:51:00,2.100000,15311.000000
75%,10.000000,2010-10-15 12:45:00,4.210000,16799.000000
max,19152.000000,2010-12-09 20:01:00,25111.090000,18287.000000
std,107.424110,NaN,146.126914,1680.811316


In [ ]:
df.isnull().sum()

,0
Invoice,0
StockCode,0
Description,2928
Quantity,0
InvoiceDate,0
Price,0
Customer ID,107927
Country,0


In [ ]:
df_cleaned = df.dropna(subset=['Description', 'Customer ID']).copy()
print("Shape of DataFrame after dropping rows with missing values:")
display(df_cleaned.shape)
print("Missing values after dropping:")
display(df_cleaned.isnull().sum())

Shape of DataFrame after dropping rows with missing values:


(417534, 8)

Missing values after dropping:


,0
Invoice,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
Price,0
Customer ID,0
Country,0


In [ ]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 417534 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      417534 non-null  object        
 1   StockCode    417534 non-null  object        
 2   Description  417534 non-null  object        
 3   Quantity     417534 non-null  int64         
 4   InvoiceDate  417534 non-null  datetime64[ns]
 5   Price        417534 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      417534 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 28.7+ MB


In [ ]:
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')

In [ ]:
print("Duplicate rows in df_cleaned before dropping:", df_cleaned.duplicated().sum())

Duplicate rows in df_cleaned before dropping: 6771


In [ ]:
df_cleaned.drop_duplicates(inplace=True)
print('Duplicate rows in df_cleaned after dropping:', df_cleaned.duplicated().sum())

Duplicate rows in df_cleaned after dropping: 0


In [ ]:
display(df_cleaned.head())

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [ ]:
df["Invoice"].astype(str).str.startswith("C").value_counts()

,count
Invoice,
False,515255
True,10206


Create transaction amount

In [ ]:
df["TotalAmount"] = df["Quantity"] * df["Price"]

In [ ]:
df[["Quantity", "Price", "TotalAmount"]].head()

,Quantity,Price,TotalAmount
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


Create our clean transaction dataset

In [ ]:
clean_df = df[
    df["Customer ID"].notna() &
    (df["Quantity"] > 0) &
    (df["Price"] > 0)
].copy()

In [ ]:
print("Original rows:", len(df))
print("Clean rows:", len(clean_df))

Original rows: 525461
Clean rows: 407664


Create customer features

In [ ]:
customer_df = clean_df.groupby("Customer ID").agg(
    total_orders=("Invoice", "nunique"),
    total_spend=("TotalAmount", "sum"),
    avg_order_value=("TotalAmount", "mean"),
    total_items=("Quantity", "sum"),
    first_purchase=("InvoiceDate", "min"),
    last_purchase=("InvoiceDate", "max")
).reset_index()

In [ ]:
customer_df["customer_tenure_days"] = (
    customer_df["last_purchase"] -
    customer_df["first_purchase"]
).dt.days

In [ ]:
customer_df = customer_df.rename(
    columns={"Customer ID": "customer_id"}
)

In [ ]:
customer_df.head()

,customer_id,total_orders,total_spend,avg_order_value,total_items,first_purchase,last_purchase,customer_tenure_days
0,12346.0,11,372.86,11.298788,70,2009-12-14 08:34:00,2010-06-28 13:53:00,196
1,12347.0,2,1323.32,18.638310,828,2010-10-31 14:20:00,2010-12-07 14:57:00,37
2,12348.0,1,222.16,11.108000,373,2010-09-27 14:59:00,2010-09-27 14:59:00,0
3,12349.0,3,2671.14,26.187647,993,2010-04-29 13:20:00,2010-10-28 08:23:00,181
4,12351.0,1,300.93,14.330000,261,2010-11-29 15:23:00,2010-11-29 15:23:00,0


Create cancellation records separately:

In [ ]:
cancelled_df = df[
    df["Invoice"].astype(str).str.startswith("C")
].copy()

Calculate cancellations per customer:

In [ ]:
cancel_count = cancelled_df.groupby("Customer ID").agg(
    cancelled_orders=("Invoice", "nunique")
).reset_index()

In [ ]:
customer_df = customer_df.merge(cancel_count,left_on="customer_id", right_on="Customer ID", how="left")
customer_df["cancelled_orders"] = (customer_df["cancelled_orders"].fillna(0))

In [ ]:
customer_df["cancellation_rate"] = (
    customer_df["cancelled_orders"] /
    customer_df["total_orders"]
)

Save our first dataset

In [ ]:
customer_df.to_csv("/content/drive/MyDrive/razorpay hackathon dataset/customers.csv",index=False)

In [ ]:
customer_df.shape

(4312, 11)

In [ ]:
customer_df.head()

,customer_id,total_orders,total_spend,avg_order_value,total_items,first_purchase,last_purchase,customer_tenure_days,Customer ID,cancelled_orders,cancellation_rate
0,12346.0,11,372.86,11.298788,70,2009-12-14 08:34:00,2010-06-28 13:53:00,196,12346.0,4.0,0.363636
1,12347.0,2,1323.32,18.638310,828,2010-10-31 14:20:00,2010-12-07 14:57:00,37,NaN,0.0,0.000000
2,12348.0,1,222.16,11.108000,373,2010-09-27 14:59:00,2010-09-27 14:59:00,0,NaN,0.0,0.000000
3,12349.0,3,2671.14,26.187647,993,2010-04-29 13:20:00,2010-10-28 08:23:00,181,12349.0,1.0,0.333333
4,12351.0,1,300.93,14.330000,261,2010-11-29 15:23:00,2010-11-29 15:23:00,0,NaN,0.0,0.000000


In [ ]:
customer_df.describe()

,customer_id,total_orders,total_spend,avg_order_value,total_items,first_purchase,last_purchase,customer_tenure_days,Customer ID,cancelled_orders,cancellation_rate
count,4312.000000,4312.000000,4312.000000,4312.000000,4312.000000,4312,4312,4312.000000,1729.000000,4312.000000,4312.000000
mean,15349.290353,4.455705,2048.238236,36.961744,1284.404917,2010-04-29 01:20:08.710575104,2010-09-10 09:08:37.138219008,133.998609,15170.704453,0.996521,0.208932
min,12346.000000,1.000000,2.950000,1.993684,1.000000,2009-12-01 07:45:00,2009-12-01 09:55:00,0.000000,12346.000000,0.000000,0.000000
25%,13882.500000,1.000000,307.987500,11.112076,158.000000,2010-01-15 11:34:30,2010-07-27 09:53:00,0.000000,13736.000000,0.000000,0.000000
50%,15350.500000,2.000000,706.020000,17.351923,382.000000,2010-04-01 08:26:00,2010-10-18 16:34:30,105.000000,15079.000000,0.000000,0.000000
75%,16834.250000,5.000000,1723.142500,24.839503,996.250000,2010-08-15 11:15:30,2010-11-22 11:01:30,254.000000,16570.000000,1.000000,0.333333
max,18287.000000,205.000000,349164.350000,10953.500000,220600.000000,2010-12-09 16:08:00,2010-12-09 20:01:00,373.000000,18287.000000,65.000000,5.000000
std,1701.200176,8.170213,8914.481280,217.711120,6459.164580,NaN,NaN,132.827183,1678.512629,2.458043,0.369752


In [ ]:
import numpy as np
np.random.seed(42)

Generate failed payment events

In [ ]:
# Number of synthetic failed payment events
n_events = 5000

# Randomly select customers
selected_customers = customer_df.sample(
    n=n_events,
    replace=True,
    random_state=42
).reset_index(drop=True)

# Create payment events
payment_events = pd.DataFrame()

payment_events["transaction_id"] = [
    f"TXN{i:06d}" for i in range(1, n_events + 1)
]

payment_events["customer_id"] = selected_customers["customer_id"]

# Generate transaction amount based on customer's average order value
payment_events["amount"] = (
    selected_customers["avg_order_value"] *
    np.random.uniform(0.6, 1.4, n_events)
).round(2)

# Prevent unrealistic zero/negative values
payment_events["amount"] = payment_events["amount"].clip(lower=100)

# Failure types
failure_types = [
    "NETWORK_ERROR",
    "TIMEOUT",
    "INSUFFICIENT_FUNDS",
    "CARD_EXPIRED",
    "BANK_DECLINED",
    "LIMIT_EXCEEDED",
    "UNKNOWN_ERROR"
]

# Probabilities for each failure type
failure_probabilities = [
    0.20,
    0.15,
    0.20,
    0.15,
    0.15,
    0.10,
    0.05
]

payment_events["failure_type"] = np.random.choice(
    failure_types,
    size=n_events,
    p=failure_probabilities
)

# First failed attempt
payment_events["attempt_number"] = 1

# Generate timestamps
start_date = pd.Timestamp("2025-01-01")
end_date = pd.Timestamp("2025-12-31")

random_days = np.random.randint(
    0,
    (end_date - start_date).days + 1,
    n_events
)

payment_events["timestamp"] = (
    start_date +
    pd.to_timedelta(random_days, unit="D")
)

payment_events.head()

,transaction_id,customer_id,amount,failure_type,attempt_number,timestamp
0,TXN000001,13590.0,100.0,INSUFFICIENT_FUNDS,1,2025-07-02
1,TXN000002,17550.0,100.0,INSUFFICIENT_FUNDS,1,2025-12-18
2,TXN000003,16635.0,100.0,LIMIT_EXCEEDED,1,2025-02-05
3,TXN000004,13037.0,100.0,TIMEOUT,1,2025-02-06
4,TXN000005,17121.0,100.0,LIMIT_EXCEEDED,1,2025-01-14


Add customer history

In [ ]:
customer_features = customer_df[
    [
        "customer_id",
        "total_orders",
        "total_spend",
        "avg_order_value",
        "customer_tenure_days",
        "cancelled_orders",
        "cancellation_rate"
    ]
].copy()

payment_events = payment_events.merge(
    customer_features,
    on="customer_id",
    how="left"
)

payment_events.head()

,transaction_id,customer_id,amount,failure_type,attempt_number,timestamp,total_orders,total_spend,avg_order_value,customer_tenure_days,cancelled_orders,cancellation_rate
0,TXN000001,13590.0,100.0,INSUFFICIENT_FUNDS,1,2025-07-02,12,4367.67,13.522198,362,2.0,0.166667
1,TXN000002,17550.0,100.0,INSUFFICIENT_FUNDS,1,2025-12-18,6,2280.48,7.809863,272,1.0,0.166667
2,TXN000003,16635.0,100.0,LIMIT_EXCEEDED,1,2025-02-05,1,473.00,18.920000,0,0.0,0.000000
3,TXN000004,13037.0,100.0,TIMEOUT,1,2025-02-06,9,3033.88,16.488478,336,4.0,0.444444
4,TXN000005,17121.0,100.0,LIMIT_EXCEEDED,1,2025-01-14,1,314.74,26.228333,0,0.0,0.000000


Check the dataset

In [ ]:
print("Rows:", payment_events.shape[0])
print("Columns:", payment_events.shape[1])

Rows: 5000
Columns: 12


In [ ]:
payment_events["failure_type"].value_counts()

,count
failure_type,
INSUFFICIENT_FUNDS,1074
NETWORK_ERROR,1014
TIMEOUT,770
CARD_EXPIRED,725
BANK_DECLINED,699
LIMIT_EXCEEDED,500
UNKNOWN_ERROR,218


In [ ]:
payment_events.describe()

,customer_id,amount,attempt_number,timestamp,total_orders,total_spend,avg_order_value,customer_tenure_days,cancelled_orders,cancellation_rate
count,5000.000000,5000.000000,5000.0,5000,5000.00000,5000.000000,5000.000000,5000.000000,5000.00000,5000.000000
mean,15397.693600,122.075366,1.0,2025-07-03 05:53:57.120000,4.36520,2259.977345,45.444201,131.510800,0.95180,0.211584
min,12347.000000,100.000000,1.0,2025-01-01 00:00:00,1.00000,2.950000,2.308947,0.000000,0.00000,0.000000
25%,13918.000000,100.000000,1.0,2025-04-02 00:00:00,1.00000,304.195000,10.844417,0.000000,0.00000,0.000000
50%,15403.500000,100.000000,1.0,2025-07-05 00:00:00,2.00000,676.170000,17.249302,104.000000,0.00000,0.000000
75%,16877.750000,100.000000,1.0,2025-10-03 00:00:00,5.00000,1699.350000,24.839503,250.000000,1.00000,0.333333
max,18287.000000,12517.660000,1.0,2025-12-31 00:00:00,155.00000,349164.350000,10953.500000,373.000000,36.00000,5.000000
std,1690.537636,354.303914,0.0,NaN,7.69551,11092.755082,357.353829,131.481516,2.13925,0.387604


In [ ]:
payment_events.isnull().sum()

,0
transaction_id,0
customer_id,0
amount,0
failure_type,0
attempt_number,0
timestamp,0
total_orders,0
total_spend,0
avg_order_value,0
customer_tenure_days,0


In [ ]:
payment_events.to_csv(
    "/content/drive/MyDrive/razorpay hackathon dataset/payment_events.csv",
    index=False
)

In [ ]:
np.random.seed(42)

interventions = {
    "NETWORK_ERROR": [
        "RETRY",
        "PAYMENT_LINK",
        "ESCALATE"
    ],
    "TIMEOUT": [
        "RETRY",
        "PAYMENT_LINK",
        "ESCALATE"
    ],
    "INSUFFICIENT_FUNDS": [
        "REMINDER",
        "PAYMENT_LINK",
        "ESCALATE"
    ],
    "CARD_EXPIRED": [
        "UPDATE_CARD",
        "PAYMENT_LINK",
        "ESCALATE"
    ],
    "BANK_DECLINED": [
        "RETRY",
        "PAYMENT_LINK",
        "ESCALATE"
    ],
    "LIMIT_EXCEEDED": [
        "PAYMENT_LINK",
        "ESCALATE"
    ],
    "UNKNOWN_ERROR": [
        "RETRY",
        "ESCALATE"
    ]
}

In [ ]:
def choose_intervention(failure_type, recovery_score):

    if failure_type == "NETWORK_ERROR":
        if recovery_score >= 0.50:
            return "RETRY"
        else:
            return "PAYMENT_LINK"

    elif failure_type == "TIMEOUT":
        if recovery_score >= 0.50:
            return "RETRY"
        else:
            return "PAYMENT_LINK"

    elif failure_type == "INSUFFICIENT_FUNDS":
        if recovery_score >= 0.60:
            return "REMINDER"
        else:
            return "PAYMENT_LINK"

    elif failure_type == "CARD_EXPIRED":
        return "UPDATE_CARD"

    elif failure_type == "BANK_DECLINED":
        if recovery_score >= 0.65:
            return "RETRY"
        else:
            return "PAYMENT_LINK"

    elif failure_type == "LIMIT_EXCEEDED":
        return "PAYMENT_LINK"

    else:
        if recovery_score >= 0.70:
            return "RETRY"
        else:
            return "ESCALATE"

In [ ]:
def calculate_customer_score(row):

    score = 0.50

    # Customer loyalty
    if row["total_orders"] >= 20:
        score += 0.15
    elif row["total_orders"] >= 10:
        score += 0.08

    # Customer tenure
    if row["customer_tenure_days"] >= 365:
        score += 0.10
    elif row["customer_tenure_days"] >= 180:
        score += 0.05

    # Cancellation behavior
    if row["cancellation_rate"] < 0.05:
        score += 0.10
    elif row["cancellation_rate"] > 0.20:
        score -= 0.10

    # Add small natural variation
    score += np.random.normal(0, 0.05)

    return np.clip(score, 0.05, 0.95)

In [ ]:
payment_events["customer_recovery_score"] = payment_events.apply(calculate_customer_score,axis=1)

In [ ]:
payment_events["customer_recovery_score"].describe()

,customer_recovery_score
count,5000.000000
mean,0.559030
std,0.102777
min,0.249618
25%,0.484258
50%,0.578408
75%,0.632052
max,0.950000


Add failure-type effect

In [ ]:
failure_effect = {
    "NETWORK_ERROR": 0.20,
    "TIMEOUT": 0.15,
    "INSUFFICIENT_FUNDS": 0.05,
    "CARD_EXPIRED": 0.10,
    "BANK_DECLINED": -0.05,
    "LIMIT_EXCEEDED": -0.10,
    "UNKNOWN_ERROR": -0.15
}

In [ ]:
payment_events["base_recovery_score"] = (
    payment_events["customer_recovery_score"]
    +
    payment_events["failure_type"].map(failure_effect)
)

Consider transaction amount

In [ ]:
payment_events["amount_ratio"] = (payment_events["amount"] /payment_events["avg_order_value"])

In [ ]:
payment_events["amount_effect"] = np.where(
    payment_events["amount_ratio"] > 2,
    -0.10,
    np.where(
        payment_events["amount_ratio"] < 0.8,
        0.03,
        0
    )
)

In [ ]:
payment_events["pre_intervention_score"] = (
    payment_events["base_recovery_score"]
    +
    payment_events["amount_effect"]
)

Add intervention effect

In [ ]:
intervention_effect = {
    "RETRY": 0.05,
    "PAYMENT_LINK": 0.08,
    "REMINDER": 0.04,
    "UPDATE_CARD": 0.15,
    "ESCALATE": 0.02
}

Select intervention

In [ ]:
payment_events["intervention"] = payment_events.apply(
    lambda row: choose_intervention(
        row["failure_type"],
        row["pre_intervention_score"]
    ),
    axis=1
)

In [ ]:
payment_events["intervention_effect"] = (
    payment_events["intervention"]
    .map(intervention_effect)
)

Calculate final recovery probability

In [ ]:
payment_events["recovery_probability"] = (
    payment_events["pre_intervention_score"]
    +
    payment_events["intervention_effect"]
)

In [ ]:
payment_events["recovery_probability"] = (
    payment_events["recovery_probability"]
    .clip(0.02, 0.97)
)

In [ ]:
payment_events["recovery_probability"].describe()

,recovery_probability
count,5000.000000
mean,0.610845
std,0.153324
min,0.074061
25%,0.511514
50%,0.624354
75%,0.724246
max,0.970000


Generate recovery outcome

In [ ]:
payment_events["recovered"] = np.random.binomial(1,payment_events["recovery_probability"])

Calculate recovered amount

In [ ]:
payment_events["amount_recovered"] = np.where(
    payment_events["recovered"] == 1,
    payment_events["amount"],
    0
)

Generate recovery time

In [ ]:
payment_events["recovery_time_hours"] = np.where(
    payment_events["recovered"] == 1,
    np.random.randint(1, 73, len(payment_events)),
    np.nan
)

In [ ]:
recovery_outcomes = payment_events[
    [
        "transaction_id",
        "intervention",
        "recovered",
        "amount_recovered",
        "recovery_time_hours"
    ]
].copy()

In [ ]:
recovery_outcomes.to_csv(
    "/content/drive/MyDrive/razorpay hackathon dataset/recovery_outcomes.csv",
    index=False
)

In [ ]:
print("Total transactions:", len(recovery_outcomes))
print("Recovered:",recovery_outcomes["recovered"].sum())
print("Recovery rate:",recovery_outcomes["recovered"].mean())
print("Revenue recovered: ₹",recovery_outcomes["amount_recovered"].sum())

Total transactions: 5000
Recovered: 3087
Recovery rate: 0.6174
Revenue recovered: ₹ 383152.96


In [ ]:
payment_events.groupby("failure_type").agg(
    transactions=("transaction_id", "count"),
    recovery_rate=("recovered", "mean"),
    revenue_recovered=("amount_recovered", "sum")
).sort_values(
    "recovery_rate",
    ascending=False
)

,transactions,recovery_rate,revenue_recovered
failure_type,,,
NETWORK_ERROR,1014,0.735700,96815.03
CARD_EXPIRED,725,0.731034,58132.22
TIMEOUT,770,0.676623,68892.26
INSUFFICIENT_FUNDS,1074,0.594041,74692.50
BANK_DECLINED,699,0.499285,50361.43
LIMIT_EXCEEDED,500,0.476000,27359.19
UNKNOWN_ERROR,218,0.298165,6900.33


In [ ]:
payment_events.groupby("intervention").agg(
    transactions=("transaction_id", "count"),
    recovery_rate=("recovered", "mean"),
    revenue_recovered=("amount_recovered", "sum")
).sort_values(
    "recovery_rate",
    ascending=False
)

,transactions,recovery_rate,revenue_recovered
intervention,,,
RETRY,1593,0.738858,156943.13
UPDATE_CARD,725,0.731034,58132.22
REMINDER,228,0.684211,19794.03
PAYMENT_LINK,2236,0.518336,141383.25
ESCALATE,218,0.298165,6900.33
